# **Import**

In [1]:
import pandas as pd
diabetes = pd.read_csv("for_label_encoding.csv")

In [2]:
diabetes.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'weight',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'payer_code',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'change',
 'diabetesMed',
 'readmitted',
 'max_glu_serum_>300',
 'max_glu_serum_MISSING_TEMP_VAL',
 'max_glu_serum_Norm',
 'A1Cresult_>8',
 'A1Cresult_MISSING_TEMP_VAL',
 'A1Cresult_Norm',
 'inpatient_discharge_interaction',
 'inpatient_admission_interaction',
 'time_med_interaction',
 'lab_med_interactio

## Combine  '<30' & '>30' classes to positive class

In [3]:
import numpy as np
diabetes['readmitted'] = np.where(diabetes['readmitted'] == 'NO', 0, 1)
print(diabetes['readmitted'].value_counts())

readmitted
0    30445
1    24764
Name: count, dtype: int64


# **Feature Engineering**

## Weight: Midpoint Adjustment, Missingness Indicator

In [4]:
for i in range(len(diabetes)): # Try dropping weight feature
    if diabetes.loc[i, "weight"] == "[0-25)":
        diabetes.loc[i, "weight"] = 12.5
    elif diabetes.loc[i, "weight"] == "[25-50)":
        diabetes.loc[i, "weight"] = 37.5
    elif diabetes.loc[i, "weight"] == "[50-75)":
        diabetes.loc[i, "weight"] = 62.5
    elif diabetes.loc[i, "weight"] == "[75-100)":
        diabetes.loc[i, "weight"] = 87.5
    elif diabetes.loc[i, "weight"] == "[100-125)":
        diabetes.loc[i, "weight"] = 112.5
    elif diabetes.loc[i, "weight"] == "[125-150)":
        diabetes.loc[i, "weight"] = 137.5
    elif diabetes.loc[i, "weight"] == "[150-175)":
        diabetes.loc[i, "weight"] = 162.5
    elif diabetes.loc[i, "weight"] == "[175-200)":
        diabetes.loc[i, "weight"] = 187.5
    elif diabetes.loc[i, "weight"] == ">200":
        diabetes.loc[i, "weight"] = 212.5

In [5]:
diabetes["weight_missing"] = diabetes["weight"].isna().astype(int)
diabetes = diabetes.drop(columns="weight")

## Diagnoses: Little Generalization

In [6]:
def diag_grouping_1(x):
    if pd.isna(x):
        return "Missing"
    x = str(x).strip()
    if x == "?" or x == "":
        return "Missing"
    if x[0] in ["V", "v"]:
        return "V_code"
    if x[0] in ["E", "e"]:
        return "E_code"

    try:
        code = float(x)
    except:
        return "Other"

    if 1 <= code <= 139:
        return "Infectious"
    if 140 <= code <= 239:
        return "Neoplasms"
    if 240 <= code <= 279:
        return "Endocrine_other"
    if 280 <= code <= 289:
        return "Blood"
    if 290 <= code <= 319:
        return "Mental"
    if 320 <= code <= 389:
        return "Neuro_sense"
    if 390 <= code <= 459:
        if 390 <= code <= 392:
            return "Circulatory_rheumatic"
        if 393 <= code <= 398:
            return "Circulatory_other_heart"
        if 401 <= code <= 405:
            return "Circulatory_htn"
        if 410 <= code <= 414:
            return "Circulatory_ischemic"
        if 415 <= code <= 417:
            return "Circulatory_pulmonary"
        if 428 <= code <= 428.99:
            return "Circulatory_hf"
        if 430 <= code <= 438:
            return "Circulatory_cerebrovascular"
        return "Circulatory_other"
    if 460 <= code <= 519:
        if 480 <= code <= 488:
            return "Respiratory_pneumonia_flu"
        if 490 <= code <= 496:
            return "Respiratory_copd_bronchitis"
        if 493 <= code <= 493.99:
            return "Respiratory_asthma"
        return "Respiratory_other"
    if 520 <= code <= 579:
        if 530 <= code <= 539:
            return "Digestive_upper"
        if 560 <= code <= 569:
            return "Digestive_intestinal"
        if 570 <= code <= 579:
            return "Digestive_liver_pancreas"
        return "Digestive_other"
    if 580 <= code <= 629:
        return "Genitourinary"
    if 630 <= code <= 679:
        return "Pregnancy"
    if 680 <= code <= 709:
        return "Skin"
    if 710 <= code <= 739:
        return "Musculoskeletal"
    if 740 <= code <= 759:
        return "Congenital"
    if 760 <= code <= 779:
        return "Perinatal"
    if 780 <= code <= 799:
        return "Symptoms"
    if 800 <= code <= 999:
        if 800 <= code <= 829:
            return "Injury_fracture"
        if 830 <= code <= 839:
            return "Injury_dislocation"
        if 850 <= code <= 854:
            return "Injury_head"
        if 860 <= code <= 869:
            return "Injury_internal"
        if 870 <= code <= 897:
            return "Injury_open_wound"
        if 920 <= code <= 924:
            return "Injury_contusion"
        if 940 <= code <= 949:
            return "Injury_burn"
        return "Injury_other"

    if 250 <= code < 251:
        return "Diabetes"

    return "Other"


for c in ["diag_1", "diag_2", "diag_3"]:
    diabetes[c] = diabetes[c].apply(diag_grouping_1)

## Medical Specialty: Adding 'Other' for Rare Cases

In [7]:
min_count = 100

counts = diabetes["medical_specialty"].value_counts()
keep = counts[counts >= min_count].index

diabetes["medical_specialty"] = diabetes["medical_specialty"].where(
    diabetes["medical_specialty"].isin(keep),
    "Other")

## Interaction Features

In [8]:
diabetes = diabetes.drop(columns = ["inpatient_discharge_interaction", "inpatient_admission_interaction"], errors='ignore')

In [9]:
diabetes["time_med_interaction"] = np.log1p(diabetes["time_med_interaction"])

# **Quick Testing**

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from lightgbm import LGBMClassifier

# Drop ID columns
diabetes = diabetes.drop(
    columns=["encounter_id", "patient_nbr"],
    errors="ignore"
)

X = diabetes.drop(columns="readmitted")
y = diabetes["readmitted"]

X = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Train LightGBM
model2 = LGBMClassifier(random_state=42)
model2.fit(X_train, y_train)

# Evaluate
pred = model2.predict(X_test)
print(classification_report(y_test, pred))

[LightGBM] [Info] Number of positive: 19811, number of negative: 24356
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032371 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1080
[LightGBM] [Info] Number of data points in the train set: 44167, number of used features: 213
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448548 -> initscore=-0.206541
[LightGBM] [Info] Start training from score -0.206541
              precision    recall  f1-score   support

           0       0.66      0.75      0.70      6089
           1       0.63      0.52      0.57      4953

    accuracy                           0.65     11042
   macro avg       0.65      0.64      0.64     11042
weighted avg       0.65      0.65      0.65     11042



# **Further Cleaning**

## Payer Code is Not Missing at Random

In [11]:
for col in ["payer_code"]:
    diabetes[col + "_missing"] = diabetes[col].isna().astype(int)

## Rate Interaction: Feature Engineering

In [12]:
diabetes["labs_per_day"] = diabetes["num_lab_procedures"] / diabetes["time_in_hospital"]
diabetes["meds_per_day"] = diabetes["num_medications"] / diabetes["time_in_hospital"]

## High Risk Discharge Feature

In [13]:
high_risk_discharge = [3, 4, 6, 7, 8, 15, 22, 23, 24]  # transitions where patients leave acute care but still require ongoing medical support or supervision

diabetes["high_risk_discharge"] = diabetes["discharge_disposition_id"].isin(
    high_risk_discharge
).astype(int)


## Label Encode discharge_dispoisiton and_id and admission_source_id

In [14]:
diabetes['discharge_disposition_id'], _ = pd.factorize(diabetes['discharge_disposition_id'])
diabetes['admission_source_id'], _ = pd.factorize(diabetes['admission_source_id'])

# **Testing**

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from lightgbm import LGBMClassifier

# Drop ID columns
diabetes = diabetes.drop(
    columns=["encounter_id", "patient_nbr"],
    errors="ignore")

X = diabetes.drop(columns="readmitted")
y = diabetes["readmitted"]

X = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Train LightGBM
model3 = LGBMClassifier(random_state=42)
model3.fit(X_train, y_train)

# Evaluate
pred = model3.predict(X_test)
print(classification_report(y_test, pred))

[LightGBM] [Info] Number of positive: 19811, number of negative: 24356
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032731 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1452
[LightGBM] [Info] Number of data points in the train set: 44167, number of used features: 217
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448548 -> initscore=-0.206541
[LightGBM] [Info] Start training from score -0.206541
              precision    recall  f1-score   support

           0       0.66      0.75      0.70      6089
           1       0.63      0.52      0.57      4953

    accuracy                           0.65     11042
   macro avg       0.64      0.64      0.64     11042
weighted avg       0.65      0.65      0.64     11042



In [29]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from lightgbm import LGBMClassifier
from imblearn.under_sampling import RandomUnderSampler

X = diabetes.drop(columns="readmitted")
y = diabetes["readmitted"]

X = pd.get_dummies(X)

rus = RandomUnderSampler(random_state=42)
X_res, y_res = rus.fit_resample(X, y)


X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)


model = LGBMClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("\nLightGBM Classification Report - Test Set")
print(classification_report(y_test, pred))

[LightGBM] [Info] Number of positive: 19811, number of negative: 19811
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.046243 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1444
[LightGBM] [Info] Number of data points in the train set: 39622, number of used features: 216
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

LightGBM Classification Report - Test Set
              precision    recall  f1-score   support

           0       0.64      0.64      0.64      4953
           1       0.64      0.64      0.64      4953

    accuracy                           0.64      9906
   macro avg       0.64      0.64      0.64      9906
weighted avg       0.64      0.64      0.64      9906



In [30]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from lightgbm import LGBMClassifier
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd

# Assuming 'diabetes' DataFrame is already loaded and preprocessed
# Drop ID columns, if they still exist
diabetes = diabetes.drop(
    columns=["encounter_id", "patient_nbr"],
    errors="ignore"
)

X = diabetes.drop(columns="readmitted")
y = diabetes["readmitted"]

# One-hot encode categorical features
X = pd.get_dummies(X)

# Apply Random Under Sampling to balance classes
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

# Split data into training (70%), validation (15%), and test (15%) sets
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.15, random_state=42, stratify=y_resampled
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=(0.15 / 0.85), random_state=42, stratify=y_train_val
) # Adjust test_size for the second split

# Train LightGBM model
model = LGBMClassifier(random_state=3)
model.fit(X_train, y_train)

# Evaluate on Validation Set
print("\nLightGBM Classification Report - Validation Set")
val_pred = model.predict(X_val)
print(classification_report(y_val, val_pred))

# Evaluate on Test Set
print("\nLightGBM Classification Report - Test Set")
test_pred = model.predict(X_test)
print(classification_report(y_test, test_pred))

[LightGBM] [Info] Number of positive: 17334, number of negative: 17334
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026215 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1434
[LightGBM] [Info] Number of data points in the train set: 34668, number of used features: 213
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

LightGBM Classification Report - Validation Set
              precision    recall  f1-score   support

           0       0.65      0.65      0.65      3715
           1       0.65      0.65      0.65      3715

    accuracy                           0.65      7430
   macro avg       0.65      0.65      0.65      7430
weighted avg       0.65      0.65      0.65      7430


LightGBM Classification Report - Test Set
              precision    recall  f1-score   support

           0       0.6